In [ ]:
import os
import random
import numpy as np
import torch

# Import configuration
from config import CONFIG

# Import data functions
from data import (
    read_fasta, read_train_terms, parse_obo, read_IA_safe,
    select_top_k_labels, filter_terms_by_chosen, prepare_label_matrix_and_embeddings,
    create_term_mappings, load_or_create_embeddings
)



# Import model functions
from models import (
    create_model, split_train_validation, create_data_loaders,
    train_model, evaluate_model, find_best_threshold
) 


# Import inference functions
from inference import streaming_inference_embeddings

# Import utility functions
from utils import propagate_labels_up_hierarchy, build_restricted_parents_map


/Users/muhsenalzzaqry/anaconda3/envs/diffdock/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
    # ------------------------------------------------------------
train_seqs = read_fasta(CONFIG["TRAIN_FASTA"])
train_terms = read_train_terms(CONFIG["TRAIN_TERMS"])
parents_map, children_map = parse_obo(CONFIG["GO_OBO"])
test_seqs = read_fasta(CONFIG["TEST_FASTA"])
train_proteins = [p for p in train_terms.keys() if p in train_seqs]
  # Propagate train labels
if CONFIG["PROPAGATE_TRAIN_LABELS"] and parents_map:
        train_terms = propagate_labels_up_hierarchy(train_proteins, train_terms, parents_map) 

[io] Read 82404 sequences from /Users/muhsenalzzaqry/Desktop/CAFA_6_BASE_LINE/cafa-6-protein-function-prediction/Train/train_sequences.fasta
[io] Read training annotations for 82405 proteins from /Users/muhsenalzzaqry/Desktop/CAFA_6_BASE_LINE/cafa-6-protein-function-prediction/Train/train_terms.tsv
[io] Parsed OBO: 40121 nodes with parents
[io] Read 224309 sequences from /Users/muhsenalzzaqry/Desktop/CAFA_6_BASE_LINE/cafa-6-protein-function-prediction/Test/testsuperset.fasta
[prep] Propagating train labels up GO graph


In [4]:
train_terms_path = os.path.join(CONFIG["EMBED_DIR"], "train_ids.npy")
train_embeds_path = os.path.join(CONFIG["EMBED_DIR"], "train_embeds.npy")

train_terms = np.load(train_terms_path, allow_pickle=True)
train_terms = np.array([term.split('|')[1] for term in train_terms])
train_embeds = np.load(train_embeds_path)
train_seqs= {term: embed for term, embed in zip(train_terms, train_embeds)}

# Load terms and embeddings
train_terms= read_train_terms(CONFIG["TRAIN_TERMS"])
parents_map, children_map = parse_obo(CONFIG["GO_OBO"])
# test_seqs = read_fasta(CONFIG["TEST_FASTA"])
train_proteins = [p for p in train_terms.keys() if p in train_seqs]
  # Propagate train labels
if CONFIG["PROPAGATE_TRAIN_LABELS"] and parents_map:
        train_terms = propagate_labels_up_hierarchy(train_proteins, train_terms, parents_map)  


test_terms_path= os.path.join(CONFIG["EMBED_DIR"], "test_ids.npy")
test_embeds_path= os.path.join(CONFIG["EMBED_DIR"],"test_embeds.npy")
# Load test terms
test_terms = np.load(test_terms_path, allow_pickle=True)
# Load test embeddings
test_embeds = np.load(test_embeds_path)
# Create dictionary mapping protein IDs to embeddings
test_seqs = {term: embed for term, embed in zip(test_terms, test_embeds)}


[io] Read training annotations for 82405 proteins from /Users/muhsenalzzaqry/Desktop/CAFA_6_BASE_LINE/cafa-6-protein-function-prediction/Train/train_terms.tsv
[io] Parsed OBO: 40121 nodes with parents
[prep] Propagating train labels up GO graph


In [5]:
chosen_terms = select_top_k_labels(train_proteins, train_terms, CONFIG["TOP_K_LABELS"]) 
train_terms = filter_terms_by_chosen(train_proteins, train_terms, chosen_terms)
    

[prep] Restricting to top-3000 GO terms
[prep] Using 3000 target GO terms


In [6]:
print(train_proteins[0]) # lists of protein to be trained
print(train_terms["Q5W0B1"]) # prtein id with corrosponding lable 
print(train_seqs["Q5W0B1"]) 
print(train_seqs["Q5W0B1"].shape) #protein id with its corrosponding embedding

Q5W0B1
['GO:0000785', 'GO:0003674', 'GO:0003682', 'GO:0003824', 'GO:0004842', 'GO:0005488', 'GO:0005515', 'GO:0005575', 'GO:0005622', 'GO:0005694', 'GO:0006275', 'GO:0008150', 'GO:0008152', 'GO:0009987', 'GO:0016567', 'GO:0016740', 'GO:0016746', 'GO:0016755', 'GO:0019219', 'GO:0019222', 'GO:0019538', 'GO:0019787', 'GO:0032446', 'GO:0036211', 'GO:0043170', 'GO:0043226', 'GO:0043228', 'GO:0043229', 'GO:0043232', 'GO:0043412', 'GO:0043687', 'GO:0044238', 'GO:0050789', 'GO:0050794', 'GO:0051052', 'GO:0060255', 'GO:0065007', 'GO:0070647', 'GO:0080090', 'GO:0110165', 'GO:0140096']
[ 0.02792 -0.0579  -0.0077  ... -0.01051  0.02795  0.03723]
(1024,)


In [ ]:
Xembeds, Y, mlb = prepare_label_matrix_and_embeddings(
    train_proteins=train_proteins,
    train_terms=train_terms,
    train_seqs=train_seqs,
    chosen_terms=chosen_terms
)

[prep] Embeddings shape: (82404, 1024)
[prep] Label matrix shape: (82404, 3000)


In [12]:
X_train, X_val, y_train, y_val = split_train_validation(Xembeds, Y, CONFIG)
train_loader = create_data_loaders(X_train, y_train, CONFIG)

[split] Train: (70043, 1024), Val: (12361, 1024)


In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[device] Using device: {device}")
    
input_dim = X_train.shape[1]
num_labels = Y.shape[1]
model = create_model(input_dim, num_labels, CONFIG, device)

[device] Using device: cpu


In [14]:
model = train_model(model, train_loader, CONFIG, device)
    

[train] Starting training...
[train] Epoch 1/10, Loss: 0.0440


KeyboardInterrupt: 

In [29]:
ia_weights = read_IA_safe(CONFIG["IA_FILE"]) 
y_val_prob = evaluate_model(model, X_val, y_val, device)
best_thresh, best_score = find_best_threshold(y_val, y_val_prob, ia_weights, mlb, CONFIG)

[eval] best_thresh 0.1 best IA-weighted F1 0.16302529621986825


In [16]:
term_to_idx, idx_to_term = create_term_mappings(mlb)
restricted_parents = build_restricted_parents_map(mlb.classes_, parents_map, term_to_idx)
    

In [17]:
test_seqs_test = read_fasta(CONFIG["TEST_FASTA"])

[io] Read 224309 sequences from /Users/muhsenalzzaqry/Desktop/CAFA_6_BASE_LINE/cafa-6-protein-function-prediction/Test/testsuperset.fasta


In [18]:
print(test_seqs_test["A0A0C5B5G6"])

MRWQEMGYIFYPRKLR


In [30]:
streaming_inference_embeddings(
    model=model,
    test_seqs=test_seqs,
    config=CONFIG,
    device=device,
    best_thresh=best_thresh,
    mlb=mlb,
    restricted_parents=restricted_parents,
    parents_map=parents_map,
)

[test] Streaming 224309 test sequences in batches of 64
[stream] processed 0 / 224309
[stream] processed 3200 / 224309
[stream] processed 6400 / 224309
[stream] processed 9600 / 224309
[stream] processed 12800 / 224309
[stream] processed 16000 / 224309
[stream] processed 19200 / 224309
[stream] processed 22400 / 224309
[stream] processed 25600 / 224309
[stream] processed 28800 / 224309
[stream] processed 32000 / 224309
[stream] processed 35200 / 224309
[stream] processed 38400 / 224309
[stream] processed 41600 / 224309
[stream] processed 44800 / 224309
[stream] processed 48000 / 224309
[stream] processed 51200 / 224309
[stream] processed 54400 / 224309
[stream] processed 57600 / 224309
[stream] processed 60800 / 224309
[stream] processed 64000 / 224309
[stream] processed 67200 / 224309
[stream] processed 70400 / 224309
[stream] processed 73600 / 224309
[stream] processed 76800 / 224309
[stream] processed 80000 / 224309
[stream] processed 83200 / 224309
[stream] processed 86400 / 224309